In [3]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd().parent
print('PROJECT ROOT==>',PROJECT_ROOT)

PROJECT ROOT==> c:\Users\aless\OneDrive\unimi_projects\multi_agent_evaluation_on_intraoral_3D_scans


#### verifichiamo dataset/3DTeethLand_landmarks_train 

In [6]:

lower_dir = PROJECT_ROOT / "dataset" / "3DTeethLand_landmarks_train" / "lower"
upper_dir = PROJECT_ROOT / "dataset" / "3DTeethLand_landmarks_train" / "upper"

#Ottieni gli ID come nomi delle cartelle
lower_ids = {p.name for p in lower_dir.iterdir() if p.is_dir()}
upper_ids = {p.name for p in upper_dir.iterdir() if p.is_dir()}

print(f"Numero ID lower: {len(lower_ids)}")
print(f"Numero ID upper: {len(upper_ids)}")

#Intersezione
intersection = lower_ids.intersection(upper_ids)
print(f"ID in comune: {len(intersection)}")

#Differenze
only_lower = lower_ids - upper_ids
only_upper = upper_ids - lower_ids

print(f"ID presenti solo in lower: {len(only_lower)}")
print(f"ID presenti solo in upper: {len(only_upper)}")

if only_lower:
    print("Solo lower:", sorted(only_lower))

if only_upper:
    print("Solo upper:", sorted(only_upper))


Numero ID lower: 120
Numero ID upper: 120
ID in comune: 85
ID presenti solo in lower: 35
ID presenti solo in upper: 35
Solo lower: ['017U3R3T', '019ZGSJR', '01ECZRY6', '01FC0082', '01FUXUMF', '01HJHZ5X', '01J24RCK', '01JHDRT4', '01JZDW8K', '26CY2H2Q', '460C7711', '5JRH5J6E', '5M23YGJN', '6I8A5049', '92NOTURU', 'BBHLWUPK', 'C3TQ47Z0', 'DKDRMRFK', 'HPVH85IJ', 'IAKXN91M', 'ITTMBOHC', 'IUIE4BYI', 'O2LXM8BS', 'Q21I52KT', 'QPYE7NOP', 'QTDZUUZV', 'S8FFW0OP', 'VCVHV931', 'X273URFW', 'X3EW8O0A', 'XKTTBEE0', 'XNNZTXK5', 'Y48DURWV', 'YNKZHRP0', 'ZGH1UT1Q']
Solo upper: ['013TXGFK', '014JUMCF', '0165W7J4', '017FHD6X', '019NJEW6', '019ZKUHV', '01A6GW4A', '01F4JV8X', '01JZ8C06', '01K17AN8', '2HVVCR7B', '2TPJ3FG9', '3SVY023X', '4A02S6L3', '4G9LHQ2X', '5J9O0LG3', '653MMYW8', '6X2UD6H6', 'AKBDPB4C', 'ANLLPLV7', 'CNUR69O9', 'DKZWON8N', 'EJWZZZRF', 'EK26QFUW', 'H5EFRXCQ', 'I3T81ZK8', 'JXVWXY0L', 'K6LK1YRK', 'MHG8FM6A', 'P744BHYG', 'SW62DGWI', 'TTW81N1R', 'VVUXTBF8', 'YBSESUN6', 'Z8HJR6YS']


#### costruzione folder scan 3d per i quali esistono i landmarks  

In [5]:
from pathlib import Path
import shutil

#Landmark directories
landmarks_root = PROJECT_ROOT / "dataset" / "3DTeethLand_landmarks_train"
lower_landmarks = landmarks_root / "lower"
upper_landmarks = landmarks_root / "upper"

#Original scans root
original_root = PROJECT_ROOT / "dataset" / "original_scans"

#Destination root
train_scans_root = PROJECT_ROOT / "dataset" / "train_scans"
train_lower = train_scans_root / "lower"
train_upper = train_scans_root / "upper"

#Create destination directories
train_lower.mkdir(parents=True, exist_ok=True)
train_upper.mkdir(parents=True, exist_ok=True)

#IDs with landmark annotations
lower_ids = {p.name for p in lower_landmarks.iterdir() if p.is_dir()}
upper_ids = {p.name for p in upper_landmarks.iterdir() if p.is_dir()}

print(f"Lower landmark IDs: {len(lower_ids)}")
print(f"Upper landmark IDs: {len(upper_ids)}")

def find_original_scan(id_name: str, arc: str):
    """
    Cerca l'ID dentro tutti i data_part_X/<arc>/<ID>/.
    Restituisce il Path della cartella se esiste, altrimenti None.
    """
    for part in original_root.iterdir():
        arc_dir = part / arc
        candidate = arc_dir / id_name
        if candidate.exists() and candidate.is_dir():
            return candidate
    return None

def copy_scans(ids, arc: str, dst_root: Path):
    """
    Copia tutte le scansioni corrispondenti agli ID forniti.
    arc ∈ {"lower", "upper"}.
    """
    missing = []

    for id_name in ids:
        src = find_original_scan(id_name, arc)
        if src is None:
            missing.append(id_name)
            continue

        dst = dst_root / id_name
        if dst.exists():
            shutil.rmtree(dst)

        shutil.copytree(src, dst)

    return missing

missing_lower = copy_scans(lower_ids, "lower", train_lower)
missing_upper = copy_scans(upper_ids, "upper", train_upper)

print("***RISULTATI ===")
print(f"Copiate lower: {len(lower_ids) - len(missing_lower)} / {len(lower_ids)}")
print(f"Copiate upper: {len(upper_ids) - len(missing_upper)} / {len(upper_ids)}")

if missing_lower:
    print("\nLower mancanti:", missing_lower)

if missing_upper:
    print("\nUpper mancanti:", missing_upper)



Lower landmark IDs: 120
Upper landmark IDs: 120
***RISULTATI ===
Copiate lower: 33 / 120
Copiate upper: 33 / 120

Lower mancanti: ['E11NEPB6', '01HJHZ5X', '019ZGSJR', 'DGUTSTAN', '5M23YGJN', '01FPTYH2', 'LHP32GAK', '019TTV1D', '2XV86U1F', '01JZDW8K', 'O2LXM8BS', '019ZMN7R', '15AIQVK8', 'LSMGKLAH', '01502VH6', 'GG5ARUVQ', '14M656LK', 'E0CUCLHY', '3JCUF41E', '87N5YSES', 'CVTHSBS5', 'ameziani', 'M7RTPNPC', '01FJT0PR', '6QJ5RART', '01ADYT70', '01JHDRT4', '01ECZRY6', '01HY2W2Z', '55EXF0WK', '01A6HAN6', 'IUIE4BYI', '01MCM1UK', '019CNYA6', 'GANLQSL1', 'LE4E8YWJ', '01EDMPPZ', '01J24RCK', '460C7711', '017U3R3T', 'IAKXN91M', 'K4BAII5F', 'ITTMBOHC', '0155EZ2V', 'CK45DVLG', '016KWDMV', '0140YFGV', '26CY2H2Q', '949XHLS5', '014ZTUSK', 'N53MDM0A', '8MTEIYKY', '9YQIGSGN', '5JRH5J6E', '51MXL2ZA', '0154PPVX', '6XSU5W4B', '0171X5EV', '82J9I2HU', '92NOTURU', '67PV9M7X', '8SLVB1AH', '01JNZKAX', 'GSHA8E4C', '7IYV2N3D', '0140W3ND', 'DKDRMRFK', 'HPVH85IJ', '01J9K9S6', 'LLJGPB8I', '01FUXUMF', 'K70NYZYE', 'HDVY

#### costruzione folder scan 3d per modello migliore (toothinstancenet) 

In [13]:
from pathlib import Path
import shutil

BASE = Path("..")

landmarks_root = BASE / "dataset" / "3DTeethLand_landmarks_train"
original_root = BASE / "dataset" / "original_scans"
tin_root = BASE / "dataset" / "toothinstancenet_input"
tin_root.mkdir(exist_ok=True)

# Landmark IDs
ids_by_arc = {
    "lower": {p.name for p in (landmarks_root / "lower").iterdir() if p.is_dir()},
    "upper": {p.name for p in (landmarks_root / "upper").iterdir() if p.is_dir()},
}

def find_scan_folder(id_name: str, arc: str):
    for part in original_root.iterdir():
        candidate = part / arc / id_name
        if candidate.is_dir():
            return candidate
    return None

def find_mesh_and_seg(scan_folder: Path, id_name: str, arc: str):
    mesh = None
    seg = None

    for f in scan_folder.iterdir():
        suf = f.suffix.lower()

        # mesh
        if suf in {".obj", ".ply", ".stl"}:
            mesh = f

        # segmentazione GT
        if f.name.lower().startswith(id_name.lower()) and f.name.lower().endswith(f"{arc}.json"):
            seg = f

    return mesh, seg

def find_landmark_gt(id_name: str, arc: str):
    kpt = landmarks_root / arc / id_name / f"{id_name}_{arc}__kpt.json"
    return kpt if kpt.exists() else None

def copy_all(id_name: str, arc: str):
    scan_folder = find_scan_folder(id_name, arc)
    if scan_folder is None:
        return False

    mesh, seg = find_mesh_and_seg(scan_folder, id_name, arc)
    kpt = find_landmark_gt(id_name, arc)

    if mesh is None or kpt is None:
        return False

    #Copia mesh rinominata
    dst_mesh = tin_root / f"{id_name}_{arc}{mesh.suffix.lower()}"
    shutil.copy(mesh, dst_mesh)

    #Copia segmentazione GT
    if seg is not None:
        shutil.copy(seg, tin_root / f"{id_name}_{arc}_seg.json")

    #Copia landmark GT
    shutil.copy(kpt, tin_root / f"{id_name}_{arc}__kpt.json")

    return True

valid = {arc: [] for arc in ids_by_arc}

for arc, ids in ids_by_arc.items():
    for id_name in ids:
        if copy_all(id_name, arc):
            valid[arc].append(id_name)

print("\n=== RISULTATI FINALI ===")
print(f"Lower pronti: {len(valid['lower'])}")
print(f"Upper pronti: {len(valid['upper'])}")
print(f"Totale scans pronte: {len(valid['lower']) + len(valid['upper'])}")



=== RISULTATI FINALI ===
Lower pronti: 33
Upper pronti: 33
Totale scans pronte: 66
